In [1]:
# Install libraries and initialize Spark
!pip -q install pyspark scikit-learn scipy

import os, zipfile, shutil, requests
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.neighbors import NearestNeighbors

spark = SparkSession.builder.appName("Book Crossing Recommender").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [2]:
# Create data lake folders and download the raw dataset
BASE = "/content/book_recommender_data_lake"
BRONZE, SILVER, GOLD = [os.path.join(BASE, x) for x in ["bronze", "silver", "gold"]]
for path in [BRONZE, SILVER, GOLD]:
    os.makedirs(path, exist_ok=True)

ZIP_PATH = "/content/book-crossings.zip"
EXTRACT = "/content/book-crossings"
URL = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"

if not os.path.exists(ZIP_PATH):
    response = requests.get(URL, timeout=120)
    response.raise_for_status()
    open(ZIP_PATH, "wb").write(response.content)

if not os.path.isdir(EXTRACT):
    os.makedirs(EXTRACT, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT)

for name in ["BX-Users.csv", "BX-Books.csv", "BX-Book-Ratings.csv"]:
    found = next((os.path.join(root, name) for root, _, files in os.walk(EXTRACT) if name in files), None)
    if found is None:
        raise FileNotFoundError(name)
    shutil.copy2(found, os.path.join(BRONZE, name))

In [3]:
# Load the raw semicolon-separated, ISO-8859-1 files and create Bronze Parquet tables
def read_raw(name):
    return (spark.read.option("header", True).option("sep", ";")
            .option("encoding", "ISO-8859-1").option("quote", '"')
            .option("escape", '"').option("multiLine", True)
            .option("inferSchema", True)
            .csv(os.path.join(BRONZE, name)))

users_raw = read_raw("BX-Users.csv")
books_raw = read_raw("BX-Books.csv")
ratings_raw = read_raw("BX-Book-Ratings.csv")

users_raw.withColumn("ingestion_timestamp", F.current_timestamp()).write.mode("overwrite").parquet(BRONZE + "/users")
books_raw.withColumn("ingestion_timestamp", F.current_timestamp()).write.mode("overwrite").parquet(BRONZE + "/books")
ratings_raw.withColumn("ingestion_timestamp", F.current_timestamp()).write.mode("overwrite").parquet(BRONZE + "/ratings")

In [4]:
# Explore Bronze tables and report EDA findings
users_b = spark.read.parquet(BRONZE + "/users")
books_b = spark.read.parquet(BRONZE + "/books")
ratings_b = spark.read.parquet(BRONZE + "/ratings")

for name, df in [("Users", users_b), ("Books", books_b), ("Ratings", ratings_b)]:
    print(f"\n{name}: {df.count()} rows, {len(df.columns)} columns")
    df.printSchema()
    df.show(5, truncate=False)
    print("Missing values:", df.select([
        F.sum(F.col(c).isNull().cast("long")).alias(c) for c in df.columns
    ]).first().asDict())

ratings_b.groupBy("Book-Rating").count().orderBy("Book-Rating").show()
print("Zero ratings:", ratings_b.filter(F.col("Book-Rating") == 0).count())
users_b.groupBy("Age").count().orderBy("Age").show(30)
books_b.groupBy("Year-Of-Publication").count().orderBy("Year-Of-Publication").show(30)
ratings_b.groupBy("User-ID").count().orderBy(F.desc("count")).show(10)
ratings_b.groupBy("ISBN").count().orderBy(F.desc("count")).show(10)

u = ratings_b.select("User-ID").distinct().count()
b = ratings_b.select("ISBN").distinct().count()
pairs = ratings_b.select("User-ID", "ISBN").distinct().count()
print("Matrix sparsity (%):", round(100 * (1 - pairs / (u * b)), 4) if u*b else 0)

# EDA actions: deduplicate, remove zero ratings, validate ages and years,
# split locations, standardise text, remove image URLs, and filter sparse users/books.


Users: 278858 rows, 4 columns
root
 |-- User-ID: integer (nullable = true)
 |-- Location: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

+-------+----------------------------------+----+--------------------------+
|User-ID|Location                          |Age |ingestion_timestamp       |
+-------+----------------------------------+----+--------------------------+
|1      |nyc, new york, usa                |NULL|2026-11-25 16:46:13.999779|
|2      |stockton, california, usa         |18  |2026-11-25 16:46:13.999779|
|3      |moscow, yukon territory, russia   |NULL|2026-11-25 16:46:13.999779|
|4      |porto, v.n.gaia, portugal         |17  |2026-11-25 16:46:13.999779|
|5      |farnborough, hants, united kingdom|NULL|2026-11-25 16:46:13.999779|
+-------+----------------------------------+----+--------------------------+
only showing top 5 rows
Missing values: {'User-ID': 0, 'Location': 0, 'Age': 0, 'ingestion_timestamp'

In [5]:
# Clean Silver tables
users = users_b.drop("ingestion_timestamp").dropDuplicates()
books = books_b.drop("ingestion_timestamp").dropDuplicates()
ratings = ratings_b.drop("ingestion_timestamp").dropDuplicates()

# Remove book image URL columns
image_cols = [c for c in books.columns if "image-url" in c.lower()]
if image_cols:
    books = books.drop(*image_cols)

# Trim text fields and standardise selected descriptive columns
def clean_strings(df, title_case=()):
    for c, t in df.dtypes:
        if t == "string":
            df = df.withColumn(c, F.trim(F.col(c)))
            df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))
            if c in title_case:
                df = df.withColumn(c, F.initcap(F.col(c)))
    return df

# Clean text fields in all three Silver tables
users = clean_strings(users)
books = clean_strings(books, ["Book-Title", "Book-Author", "Publisher"])
ratings = clean_strings(ratings)

# Standardise ISBN values so ratings match the book records
books = books.withColumn("ISBN", F.upper(F.trim(F.col("ISBN"))))
ratings = ratings.withColumn("ISBN", F.upper(F.trim(F.col("ISBN"))))

# Remove records with missing key identifiers
books = books.filter(F.col("ISBN").isNotNull() & (F.col("ISBN") != ""))
ratings = ratings.filter(
    F.col("ISBN").isNotNull() &
    (F.col("ISBN") != "") &
    F.col("User-ID").isNotNull()
)

# Clean user ages and fill missing values with the median
users = users.withColumn(
    "Age",
    F.expr("""
        CASE
            WHEN UPPER(TRIM(CAST(Age AS STRING))) IN ('NULL', 'NA', 'N/A', '')
            THEN NULL
            ELSE TRY_CAST(TRIM(CAST(Age AS STRING)) AS DOUBLE)
        END
    """)
)

users = users.withColumn(
    "Age",
    F.when((F.col("Age") < 5) | (F.col("Age") > 100), None)
     .otherwise(F.col("Age"))
)

med = users.approxQuantile("Age", [0.5], 0.01)

if med:
    users = users.fillna({"Age": float(med[0])})

# Split Location safely into city, state, and country
parts = F.split(F.coalesce(F.col("Location"), F.lit("")), ",")

users = (
    users
    .withColumn("City", F.trim(F.get(parts, 0)))
    .withColumn("State", F.trim(F.get(parts, 1)))
    .withColumn("Country", F.trim(F.get(parts, 2)))
    .drop("Location")
)
for c in ["City", "State", "Country"]:
    users = users.withColumn(c, F.when((F.col(c) == "") | F.col(c).isNull(), None).otherwise(F.initcap(F.col(c))))

# Remove invalid publication years and keep explicit ratings from 1 to 10
# Convert invalid publication years to NULL
books = books.withColumn(
    "Year-Of-Publication",
    F.expr("""
        TRY_CAST(TRIM(CAST(`Year-Of-Publication` AS STRING)) AS INT)
    """)
)

# Remove zero, negative, and future publication years
books = books.withColumn(
    "Year-Of-Publication",
    F.when(
        (F.col("Year-Of-Publication") <= 0) |
        (F.col("Year-Of-Publication") > F.year(F.current_date())),
        None
    ).otherwise(F.col("Year-Of-Publication"))
)
books = books.withColumn("Year-Of-Publication",
    F.when((F.col("Year-Of-Publication") <= 0) | (F.col("Year-Of-Publication") > F.year(F.current_date())), None)
     .otherwise(F.col("Year-Of-Publication")))
ratings = ratings.withColumn("Book-Rating", F.col("Book-Rating").cast("int")).filter(F.col("Book-Rating").between(1, 10))

users = users.withColumn("User-ID", F.col("User-ID").cast("int"))
ratings = ratings.withColumn("User-ID", F.col("User-ID").cast("int"))
books = books.withColumn("ISBN", F.col("ISBN").cast("string"))
ratings = ratings.withColumn("ISBN", F.col("ISBN").cast("string"))

In [6]:
# Keep only ratings that have a matching user and book
MIN_RATINGS = 10

# Standardise ISBN values
books = books.withColumn("ISBN", F.upper(F.trim(F.col("ISBN"))))
ratings = ratings.withColumn("ISBN", F.upper(F.trim(F.col("ISBN"))))

# Remove ratings with missing ISBN or User-ID
ratings = ratings.filter(
    F.col("ISBN").isNotNull() &
    (F.col("ISBN") != "") &
    F.col("User-ID").isNotNull()
)

# First, remove ratings whose ISBN does not exist in books
ratings = ratings.join(
    books.select("ISBN").distinct(),
    "ISBN",
    "left_semi"
)

# Remove ratings whose User-ID does not exist in users
ratings = ratings.join(
    users.select("User-ID").distinct(),
    "User-ID",
    "left_semi"
)

# Keep users with at least 10 ratings
active_users = (
    ratings.groupBy("User-ID")
    .count()
    .filter(F.col("count") >= MIN_RATINGS)
    .select("User-ID")
)

# Keep books with at least 10 ratings
active_books = (
    ratings.groupBy("ISBN")
    .count()
    .filter(F.col("count") >= MIN_RATINGS)
    .select("ISBN")
)

# Filter ratings using the minimum rating threshold
ratings = ratings.join(active_users, "User-ID", "left_semi")
ratings = ratings.join(active_books, "ISBN", "left_semi")

# Keep only users and books present in the final ratings
users = users.join(
    ratings.select("User-ID").distinct(),
    "User-ID",
    "left_semi"
)

books = books.join(
    ratings.select("ISBN").distinct(),
    "ISBN",
    "left_semi"
)

# Display Silver layer counts
print("Minimum ratings threshold:", MIN_RATINGS)
print("Silver users:", users.count())
print("Silver books:", books.count())
print("Silver ratings:", ratings.count())

# Check relationships
unmatched_books = ratings.join(
    books.select("ISBN").distinct(),
    "ISBN",
    "left_anti"
).count()

unmatched_users = ratings.join(
    users.select("User-ID").distinct(),
    "User-ID",
    "left_anti"
).count()

print("Unmatched book ISBNs:", unmatched_books)
print("Unmatched user IDs:", unmatched_users)

# Verify relationships
assert unmatched_books == 0
assert unmatched_users == 0

# Remove previous Silver output
import shutil

for folder in ["users", "books", "ratings"]:
    shutil.rmtree(SILVER + "/" + folder, ignore_errors=True)

# Save Silver tables
users.write.mode("overwrite").parquet(SILVER + "/users")
books.write.mode("overwrite").parquet(SILVER + "/books")
ratings.write.mode("overwrite").parquet(SILVER + "/ratings")

print("Silver tables saved successfully.")

Minimum ratings threshold: 10
Silver users: 6319
Silver books: 5551
Silver ratings: 87233
Unmatched book ISBNs: 0
Unmatched user IDs: 0
Silver tables saved successfully.


In [7]:
# Build Gold aggregates, lookup tables, and sparse book-user rating matrix
users_s = spark.read.parquet(SILVER + "/users")
books_s = spark.read.parquet(SILVER + "/books")
ratings_s = spark.read.parquet(SILVER + "/ratings")

user_agg = ratings_s.groupBy("User-ID").agg(F.avg("Book-Rating").alias("average_rating"), F.count("*").alias("number_of_ratings"))
book_agg = ratings_s.groupBy("ISBN").agg(F.avg("Book-Rating").alias("average_rating"), F.count("*").alias("number_of_ratings"))
book_lookup = books_s.select("ISBN", "Book-Title", "Book-Author").dropDuplicates(["ISBN"])

user_idx = (ratings_s.select("User-ID").distinct()
            .withColumn("user_idx", F.row_number().over(Window.orderBy("User-ID")) - 1))
book_idx = (ratings_s.select("ISBN").distinct()
            .withColumn("book_idx", F.row_number().over(Window.orderBy("ISBN")) - 1))
indexed = ratings_s.join(user_idx, "User-ID").join(book_idx, "ISBN")

num_users, num_books = user_idx.count(), book_idx.count()
rows = indexed.select("book_idx", "user_idx", "Book-Rating").collect()
matrix = csr_matrix(
    ([float(x["Book-Rating"]) for x in rows],
     ([int(x["book_idx"]) for x in rows], [int(x["user_idx"]) for x in rows])),
    shape=(num_books, num_users)
)
save_npz(GOLD + "/book_user_rating_matrix.npz", matrix)

user_idx.write.mode("overwrite").parquet(GOLD + "/user_index")
book_idx.write.mode("overwrite").parquet(GOLD + "/book_index")
book_lookup.write.mode("overwrite").parquet(GOLD + "/book_lookup")
user_agg.write.mode("overwrite").parquet(GOLD + "/user_aggregates")
book_agg.write.mode("overwrite").parquet(GOLD + "/book_aggregates")
print("Gold matrix shape:", matrix.shape, "Non-zero ratings:", matrix.nnz)

Gold matrix shape: (5440, 6319) Non-zero ratings: 87233


In [8]:
# Load Gold data and fit item-based cosine similarity model
matrix = load_npz(GOLD + "/book_user_rating_matrix.npz")
book_index_pd = spark.read.parquet(GOLD + "/book_index").toPandas()
user_index_pd = spark.read.parquet(GOLD + "/user_index").toPandas()
lookup_pd = spark.read.parquet(GOLD + "/book_lookup").toPandas()

metadata = book_index_pd.merge(lookup_pd, on="ISBN", how="left")
metadata["Book-Title"] = metadata["Book-Title"].fillna("Unknown title")
metadata["Book-Author"] = metadata["Book-Author"].fillna("Unknown author")
book_by_idx = metadata.set_index("book_idx").to_dict("index")
user_to_idx = dict(zip(user_index_pd["User-ID"], user_index_pd["user_idx"]))

model = NearestNeighbors(metric="cosine", algorithm="brute",
                         n_neighbors=min(11, matrix.shape[0]))
model.fit(matrix)

# Return the 10 most similar books for a given title
def similar_books(title, n=10):
    found = metadata[metadata["Book-Title"].str.casefold() == str(title).strip().casefold()]
    if found.empty:
        found = metadata[metadata["Book-Title"].str.contains(str(title), case=False, na=False, regex=False)]
    if found.empty:
        return pd.DataFrame(columns=["Book-Title", "Book-Author", "similarity"])
    idx = int(found.iloc[0]["book_idx"])
    distances, indices = model.kneighbors(matrix[idx], n_neighbors=min(n + 1, matrix.shape[0]))
    result = []
    for distance, other in zip(distances[0], indices[0]):
        if int(other) != idx:
            item = book_by_idx[int(other)]
            result.append({"Book-Title": item["Book-Title"], "Book-Author": item["Book-Author"],
                           "similarity": float(1 - distance)})
        if len(result) == n:
            break
    return pd.DataFrame(result)

# Recommend unseen books using similarity weighted by the user's rating
def recommend(user_id, n=10):
    if user_id not in user_to_idx:
        return pd.DataFrame(columns=["Book-Title", "Book-Author", "score"])
    col = int(user_to_idx[user_id])
    rated = matrix[:, col].tocoo()
    seen = set(int(x) for x in rated.row)
    liked = [(int(i), float(r)) for i, r in zip(rated.row, rated.data) if r >= 8]
    if not liked:
        liked = [(int(i), float(r)) for i, r in zip(rated.row, rated.data)]
    scores = {}
    for source, rating in liked:
        distances, indices = model.kneighbors(matrix[source], n_neighbors=min(51, matrix.shape[0]))
        for distance, candidate in zip(distances[0], indices[0]):
            candidate = int(candidate)
            if candidate not in seen and candidate != source:
                scores[candidate] = scores.get(candidate, 0) + float(1 - distance) * rating
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]
    return pd.DataFrame([{
        "Book-Title": book_by_idx[i]["Book-Title"],
        "Book-Author": book_by_idx[i]["Book-Author"],
        "score": score
    } for i, score in ranked])

In [9]:
# Demonstrate similar books and recommendations for three users
display(similar_books(metadata["Book-Title"].iloc[0], n=10))

demo_users = (ratings_s.groupBy("User-ID").count()
              .orderBy(F.desc("count")).limit(3).toPandas()["User-ID"].tolist())

for uid in demo_users:
    print("\\nUser ID:", uid)
    print("Books rated highly:")
    high = (ratings_s.filter((F.col("User-ID") == uid) & (F.col("Book-Rating") >= 8))
            .join(book_lookup, "ISBN", "left")
            .select("Book-Title", "Book-Author", "Book-Rating")
            .orderBy(F.desc("Book-Rating")).limit(10).toPandas())
    display(high)
    print("Recommendations:")
    display(recommend(int(uid), n=10))

# Final sanity checks
assert ratings_s.filter(
    ~F.col("Book-Rating").between(1, 10)
).count() == 0

assert ratings_s.join(
    users_s.select("User-ID"),
    "User-ID",
    "left_anti"
).count() == 0

assert ratings_s.join(
    books_s.select("ISBN"),
    "ISBN",
    "left_anti"
).count() == 0

assert matrix.shape == (num_books, num_users)

print("All sanity checks passed.")

,Book-Title,Book-Author,similarity
0,Oceano Mare,Alessandro Baricco,0.707107
1,Due Di Due (bestsellers),Andrea De Carlo,0.707107
2,"Novocento, Un Monologo",Alessandro Baricco,0.707107
3,Jade Peony,Wayson Choy,0.707107
4,"Eine Frau, Eine Wohnung, Ein Roman",Wilhelm Genazino,0.707107
5,Free,Paul Vincent,0.707107
6,The Girls' Guide To Hunting And Fishing,Melissa Bank,0.613572
7,Bachelor Brothers' Bed &amp; Breakfast,Bill Richardson,0.562544
8,Flight,Jan Burke,0.552158
9,God-shaped Hole,Tiffanie Debartolo,0.547176


\nUser ID: 11676
Books rated highly:


,Book-Title,Book-Author,Book-Rating
0,Cold Sassy Tree,Olive Ann Burns,10
1,My Side Of The Mountain,Jean Craighead George,10
2,"Prentice Alvin (tales Of Alvin Maker, Book 3)",Orson Scott Card,10
3,Paradise,Judith Mcnaught,10
4,A Woman Betrayed,Barbara Delinsky,10
5,Join Me,Danny Wallace,10
6,The Spirit Catches You And You Fall Down,Anne Fadiman,10
7,The Far Side Gallery 5,Gary Larson,10
8,"Faith Of The Fallen (sword Of Truth, Book 6)",Terry Goodkind,10
9,The Gift,Danielle Steel,10


Recommendations:


,Book-Title,Book-Author,score
0,Courting Trouble,Lisa Scottoline,99.569418
1,The House On Hope Street,Danielle Steel,50.578702
2,Private Scandals,Nora Roberts,48.763151
3,A Beautiful Mind : A Biography Of John Forbes ...,Sylvia Nasar,48.228173
4,Il Piccolo Principe,Antoine De Saint-exupery,43.762249
5,Tears Of The Moon (irish Trilogy),Nora Roberts,37.589217
6,Public Secrets,Nora Roberts,37.328579
7,A Perfect Stranger,Danielle Steel,36.815709
8,Gentle Warrior,Julie Garwood,36.744312
9,Once Upon A Rose,Nora Roberts,36.577535


\nUser ID: 16795
Books rated highly:


... (4 outputs left out)

,Book-Title,Book-Author,Book-Rating
0,Key Of Knowledge (key Trilogy (paperback)),Nora Roberts,10
1,The Face,Dean Koontz,10
2,To The Nines: A Stephanie Plum Novel,Janet Evanovich,10
3,Up Country: A Novel,Nelson Demille,10
4,Odd Thomas,Dean Koontz,10
5,A Place Of Hiding,Elizabeth George,10
6,"Firestorm (johansen, Iris)",Iris Johansen,10
7,Glorious Appearing: The End Of Days (left Behi...,Tim F. Lahaye,10
8,"Key Of Valor (roberts, Nora. Key Trilogy, 3.)",Nora Roberts,10
9,The Five People You Meet In Heaven,Mitch Albom,10


Recommendations:


,Book-Title,Book-Author,score
0,Free,Paul Vincent,221.064060
1,"Eine Frau, Eine Wohnung, Ein Roman",Wilhelm Genazino,220.325929
2,Jade Peony,Wayson Choy,220.325929
3,"Novocento, Un Monologo",Alessandro Baricco,220.325929
4,Due Di Due (bestsellers),Andrea De Carlo,220.325929
5,Oceano Mare,Alessandro Baricco,218.739748
6,Bachelor Brothers' Bed &amp; Breakfast,Bill Richardson,121.072830
7,Flight,Jan Burke,110.171557
8,God-shaped Hole,Tiffanie Debartolo,98.315983
9,"Schnee, Der Auf Zedern Fã?â¤llt.",David Guterson,65.754640


All sanity checks passed.
